# 01 - Text exploration

This notebook walks through the synthetic newsroom corpus that ships with the package and the preprocessing primitives in `newsroom.preprocessing`.

Everything here runs offline in a few seconds.

In [ ]:
import pandas as pd

from newsroom.data.corpus import generate_corpus

df = generate_corpus(n_docs=1500, seed=42)
print(df.shape)
df.head()

## Label balance and document length

The generator balances categories exactly; document length varies with the number of sampled sentences.

In [ ]:
print(df["label"].value_counts())

df["n_words"] = df["text"].str.split().str.len()
df.groupby("label")["n_words"].describe()[["mean", "min", "max"]]

In [ ]:
from newsroom.visualization.plots import plot_class_distribution
from IPython.display import Image

path = plot_class_distribution(df["label"], "../artifacts/notebooks/class_distribution.png")
Image(filename=str(path))

## Preprocessing step by step

`clean_text` normalises Unicode, strips HTML, replaces URLs/e-mails/numbers and expands contractions. `preprocess` adds tokenisation, stopword removal and stemming.

In [ ]:
from newsroom.preprocessing.text import clean_text, preprocess, tokenize, PreprocessConfig

raw = "Café owners can't keep up: sales rose 12.5% (see https://example.com) — “remarkable”, said the CFO."
print(clean_text(raw, numbers=True))
print(tokenize(clean_text(raw)))
print(preprocess(raw, PreprocessConfig(replace_numbers=True)))

## Most frequent stems per category

A quick look at which stems dominate each label after preprocessing.

In [ ]:
from collections import Counter

for label, group in df.groupby("label"):
    counter = Counter(tok for text in group["text"].head(200) for tok in preprocess(text))
    print(f"{label:<11}", [w for w, _ in counter.most_common(8)])

## Keywords and summaries

TextRank keywords and an extractive summary for a single article.

In [ ]:
from newsroom.keywords.textrank import TextRankExtractor
from newsroom.summarization.extractive import ExtractiveSummarizer

article = df.loc[df["label"] == "science", "text"].iloc[0]
print(article, "\n")
print("Keywords:", [k.text for k in TextRankExtractor().extract(article, top_k=6)])
print("Summary :", ExtractiveSummarizer("centroid").summarize(article, n_sentences=2).text)

## Near-duplicate scan

Template-based generation occasionally produces highly similar articles; the MinHash detector finds them without any pairwise comparison of the full corpus.

In [ ]:
from newsroom.similarity.dedup import NearDuplicateDetector

detector = NearDuplicateDetector(threshold=0.6)
detector.add_many(df["doc_id"].tolist(), df["text"].tolist())
pairs = detector.find_duplicates()
print(len(pairs), "near-duplicate pairs")
pairs[:5]